# Exercise 2 - Fine-tuning completo di DistilBERT

Kernel usato: `DLA2026-transformers`.

Obiettivo: passare dalla baseline stabile del notebook 1 al fine-tuning supervisionato di DistilBERT con una testa di classificazione per sentiment analysis.

In questo notebook verifichiamo:

1. tokenizzazione degli split con `Dataset.map`;
2. modello `AutoModelForSequenceClassification` con classification head;
3. setup completo di `Trainer`, `TrainingArguments`, `DataCollatorWithPadding` e metriche Scikit-learn;
4. training per 3 epoche e valutazione finale su validation e test.

> **Execution note**
>
> Gli output visibili sono stati prodotti durante le esecuzioni finali o di validazione del laboratorio. Nella versione di consegna i training costosi sono disattivati di default quando sono controllati da flag; checkpoint e artefatti salvati vengono usati per consultazione rapida.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

WindowsPath('DLA_2')

In [2]:
from dataclasses import asdict
import importlib
import math

import pandas as pd
import torch

import dla_lab2.sentiment as sentiment
from dla_lab2.paths import output_dir
from dla_lab2.seed import set_seed

# Ricarichiamo il modulo per usare l'ultima versione degli helper Trainer.
importlib.reload(sentiment)

from dla_lab2.sentiment import (
    build_trainer,
    build_training_arguments,
    disable_notebook_progress_callback,
    load_rotten_tomatoes,
    load_sequence_classifier,
    load_distilbert_base,
    tokenize_dataset_splits,
)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# Il fine-tuning completo deve essere abilitato esplicitamente.
RUN_FULL_FINETUNING = False


'cuda'

## Exercise 2.1 - Token Preprocessing

Richiesta dell'esercizio: tokenizzare gli split una sola volta con Hugging Face `Dataset.map`, evitando di ritokenizzare a ogni batch.

Il padding non viene fatto nella tokenizzazione: lo fara' `DataCollatorWithPadding` durante la costruzione dei batch. Questo e' piu' efficiente perche' ogni batch viene paddato solo alla lunghezza massima presente nel batch stesso.

In [3]:
if RUN_FULL_FINETUNING:
    # Carichiamo dataset e tokenizer.
    ds_dict = load_rotten_tomatoes()
    tokenizer, _ = load_distilbert_base(device=None)

    # Dataset.map applica il tokenizer a train, validation e test.
    tokenized = tokenize_dataset_splits(ds_dict, tokenizer, max_length=256)

    # Verifica richiesta: ogni elemento contiene text, label, input_ids e attention_mask.
    tokenized["train"][0].keys()
else:
    ds_dict = tokenizer = tokenized = None
    print("Dataset e tokenizer non caricati in modalita' rapida.")


<local-conda-env>\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5335.25it/s]
DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Map: 100%|██████████| 1066/1066 [00:00<00:00, 10615.14 examples/s]


dict_keys(['label', 'input_ids', 'attention_mask', 'text', 'token_type_ids'])

In [4]:
if RUN_FULL_FINETUNING:
    # Ispezioniamo un esempio tokenizzato.
    # La run mostra che text e label sono rimasti disponibili,
    # mentre input_ids e attention_mask sono tensori PyTorch.
    example = tokenized["train"][0]
    print(example["text"])
    print(example["label"])
    print(example["input_ids"].shape)
    print(example["attention_mask"].shape)
else:
    print("Esempio tokenizzato disponibile nell'output eseguito.")


the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
tensor(1)
torch.Size([47])
torch.Size([47])


Osservazioni Exercise 2.1:

- `Dataset.map` e' stato eseguito su tutti e tre gli split.
- Ogni esempio mantiene `text` e `label`.
- Ogni esempio aggiunge `input_ids` e `attention_mask`.
- Nella run, il primo esempio ha 47 token prima del padding dinamico di batch.
- La colonna `token_type_ids` compare per compatibilita' del tokenizer, ma per DistilBERT non e' centrale in questo esercizio.

## Exercise 2.2 - Model for Sequence Classification

Richiesta dell'esercizio: preparare DistilBERT per un task di sequence classification.

Usiamo `AutoModelForSequenceClassification`, che carica il backbone DistilBERT pre-addestrato e aggiunge una testa di classificazione binaria. I pesi della testa (`pre_classifier` e `classifier`) sono nuovi e quindi devono essere addestrati sul dataset Rotten Tomatoes.

In [5]:
if RUN_FULL_FINETUNING:
    # Istanza del modello con testa di classificazione binaria.
    # I pesi MISSING nel report sono attesi: corrispondono alla nuova testa di classificazione.
    classifier_model = load_sequence_classifier(num_labels=2)
    classifier_model
else:
    classifier_model = None
    print("Modello di classificazione non caricato in modalita' rapida.")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2372.93it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## Exercise 2.3 - Fine-tuning con Trainer

Richieste dell'esercizio:

1. usare `DataCollatorWithPadding` per creare batch con padding dinamico;
2. definire metriche di valutazione da logits e labels;
3. impostare `TrainingArguments` ragionevoli;
4. creare un `Trainer` con train/validation split;
5. chiamare `trainer.train()` e poi `trainer.evaluate()`.

Queste parti sono implementate negli helper:

- `build_trainer()` crea `DataCollatorWithPadding`, collega `compute_sklearn_metrics` e istanzia `Trainer`;
- `compute_sklearn_metrics` calcola accuracy, F1, precision e recall con Scikit-learn;
- `build_training_arguments()` imposta learning rate, batch size, epoche, checkpoint per epoca, scheduler cosine e best checkpoint su accuracy.

Nota tecnica: rimuoviamo il `NotebookProgressCallback` per evitare l'errore `on_train_begin must be called before on_evaluate`. Per rendere comunque chiara la run, aggiungiamo sotto una tabella esplicita con le metriche per epoca.

In [6]:
if RUN_FULL_FINETUNING:
    def model_init():
        # Ogni run parte da una nuova istanza pulita del modello pre-addestrato.
        return load_sequence_classifier(num_labels=2)


    num_epochs = 3
    batch_size = 32
    total_steps = math.ceil(len(tokenized["train"]) / batch_size) * num_epochs
    warmup_steps = math.ceil(total_steps * 0.1)

    training_args = build_training_arguments(
        output_dir=output_dir("distilbert_full_finetuning"),
        learning_rate=2e-5,
        train_batch_size=batch_size,
        eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
    )

    trainer = build_trainer(
        tokenizer=tokenizer,
        training_args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        model_init=model_init,
    )

    print(f"Total training steps: {total_steps}")
    print(f"Warmup steps: {warmup_steps}")
else:
    trainer = None
    print("Trainer non inizializzato in modalita' rapida.")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2961.35it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total training steps: 801
Warmup steps: 81


In [7]:
if RUN_FULL_FINETUNING:
    # Avvio del fine-tuning completo di DistilBERT.
    # Il callback grafico del notebook e' disattivato, quindi il riepilogo leggibile
    # delle epoche viene costruito nella cella successiva da trainer.state.log_history.
    train_output = trainer.train()
    train_output
else:
    train_output = None
    print("Fine-tuning completo non eseguito.")


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3491.09it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerN

TrainOutput(global_step=801, training_loss=0.30517593970273466, metrics={'train_runtime': 234.4535, 'train_samples_per_second': 109.147, 'train_steps_per_second': 3.416, 'total_flos': 348266105180592.0, 'train_loss': 0.30517593970273466, 'epoch': 3.0})

In [8]:
if RUN_FULL_FINETUNING:
    # Riepilogo leggibile del training.
    # `loss_log` mostra la training loss registrata ogni 50 step.
    # `epoch_eval_log` mostra le metriche validation alla fine di ogni epoca.
    history = pd.DataFrame(trainer.state.log_history)

    loss_log = history[history["loss"].notna()][["epoch", "step", "loss", "learning_rate", "grad_norm"]].copy()
    epoch_eval_log = history[history["eval_loss"].notna()][
        ["epoch", "step", "eval_loss", "eval_accuracy", "eval_f1", "eval_precision", "eval_recall"]
    ].copy()

    display(loss_log)
    display(epoch_eval_log)
else:
    display(pd.read_csv(PROJECT_ROOT / "results" / "full_finetuning_history.csv"))


,epoch,step,loss,learning_rate,grad_norm
0,0.187266,50,0.689260,1.209877e-05,0.862539
1,0.374532,100,0.540151,1.996917e-05,6.918018
2,0.561798,150,0.429496,1.956305e-05,5.067037
3,0.749064,200,0.390760,1.870356e-05,3.500590
4,0.936330,250,0.357328,1.743145e-05,3.995320
6,1.123596,300,0.316926,1.580703e-05,4.771597
7,1.310861,350,0.257733,1.390731e-05,3.643249
8,1.498127,400,0.250612,1.182236e-05,9.173053
9,1.685393,450,0.276805,9.651005e-06,6.382890
10,1.872659,500,0.263451,7.496200e-06,4.984672


,epoch,step,eval_loss,eval_accuracy,eval_f1,eval_precision,eval_recall
5,1.0,267,0.369405,0.836773,0.844086,0.807890,0.883677
11,2.0,534,0.357698,0.846154,0.839216,0.878850,0.803002
18,3.0,801,0.387215,0.852720,0.851747,0.857414,0.846154


In [9]:
if RUN_FULL_FINETUNING:
    # Valutiamo esplicitamente il best model su validation e test.
    # La funzione sotto rimuove il callback notebook anche se il Trainer era stato creato
    # in una sessione precedente.
    disable_notebook_progress_callback(trainer)
    validation_metrics = trainer.evaluate(tokenized["validation"])
    test_metrics = trainer.evaluate(tokenized["test"])

    pd.DataFrame([validation_metrics, test_metrics], index=["validation", "test"])
else:
    display(pd.read_csv(PROJECT_ROOT / "results" / "sentiment_results.csv").query("method == 'Full DistilBERT fine-tuning'"))


,eval_loss,eval_accuracy,eval_f1,eval_precision,eval_recall,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
validation,0.387215,0.852720,0.851747,0.857414,0.846154,3.1147,342.251,10.916,3.0
test,0.439009,0.844278,0.842803,0.850860,0.834897,2.7471,388.040,12.377,3.0


## Conclusioni Exercise 2

Tutti i punti richiesti sono stati svolti.

Exercise 2.1:
- gli split sono stati tokenizzati con `Dataset.map`;
- ogni elemento contiene `text`, `label`, `input_ids` e `attention_mask`;
- il padding e' lasciato al data collator, quindi avviene dinamicamente per batch.

Exercise 2.2:
- DistilBERT e' stato istanziato come `AutoModelForSequenceClassification`;
- la testa di classificazione binaria e' nuova e viene addestrata sul task;
- i messaggi `MISSING` per `pre_classifier` e `classifier` sono attesi.

Exercise 2.3:
- `DataCollatorWithPadding` e' usato dentro `build_trainer()`;
- le metriche sono calcolate con Scikit-learn da logits e labels;
- `TrainingArguments` usa 3 epoche, batch size 32, learning rate `2e-5`, weight decay `0.01`, scheduler cosine e warmup di 81 step;
- `Trainer` e' stato addestrato per 3 epoche;
- validation e test sono stati valutati esplicitamente.

Risultati osservati:

- validation accuracy: `0.8527`, F1: `0.8517`;
- test accuracy: `0.8443`, F1: `0.8428`;
- train loss media finale riportata da `TrainOutput`: `0.3052`;
- training runtime: circa `234 s`;
- best checkpoint scelto per accuracy: epoca 3.

Confronto con la baseline del notebook 1:

- baseline SVM test accuracy: `0.7946`;
- fine-tuning DistilBERT test accuracy: `0.8443`;
- miglioramento assoluto: circa `+0.0497` punti di accuracy.

Interpretazione: il fine-tuning completo migliora chiaramente la baseline congelata. La validation loss e' minima all'epoca 2 ma l'accuracy migliore arriva all'epoca 3; siccome il criterio scelto e' accuracy, il checkpoint finale e' coerente con la configurazione. Il risultato e' adatto come riferimento principale per gli esercizi successivi su fine-tuning efficiente.

## Referenced functions and source files

| Function/class | Defined in | Purpose |
| --- | --- | --- |
| `load_rotten_tomatoes` | `src/dla_lab2/sentiment.py` | Caricamento dataset sentiment. |
| `extract_cls_features_with_pipeline` | `src/dla_lab2/sentiment.py` | Feature extraction da DistilBERT. |
| `build_trainer` | `src/dla_lab2/sentiment.py` | Configurazione Hugging Face Trainer. |
| `CLIPAdapter` | `src/dla_lab2/clip_utils.py` | Adattatore leggero per feature CLIP. |
